# 06_schema_drift_test
Builds a small drifted sample file (one new column, one value with the wrong type), runs it through Raw-to-Bronze and Bronze-to-Silver, and shows that the pipeline evolves the schema and quarantines the bad row instead of crashing.

In [0]:
import json, os

LANDING_DIR = "/Volumes/workspace/default/steam"
SOURCE_FILE = f"{LANDING_DIR}/steamspy_2026-10-09.jsonl"
TEST_DIR = f"{LANDING_DIR}/drift_test"
TEST_FILE = f"{TEST_DIR}/steamspy_drift_sample.jsonl"
TEST_DATE = "2026-10-01"

os.makedirs(TEST_DIR, exist_ok=True)
with open(SOURCE_FILE) as f:
    rows = [json.loads(next(f)) for _ in range(5)]

for r in rows:
    r["genre"] = "Action"              # drift 1: a column the schema doesn't know
rows[0]["positive"] = "not_a_number"   # drift 2: a number arrives as text

with open(TEST_FILE, "w") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")

print(TEST_FILE)
for r in rows:
    print(r["appid"], r["name"], "| positive:", r["positive"], "| genre:", r["genre"])

In [0]:
spark.sql("USE workspace.steam")
SAMPLE_APPIDS = "730, 1172470, 578080, 1623730, 440"

print("genre column in bronze_steam:", "genre" in spark.table("bronze_steam").columns)
display(spark.sql("""
    SELECT (SELECT COUNT(*) FROM bronze_steam)      AS bronze_rows,
           (SELECT COUNT(*) FROM silver_steam)      AS silver_rows,
           (SELECT COUNT(*) FROM silver_quarantine) AS quarantine_rows,
           (SELECT COUNT(*) FROM schema_drift_log)  AS drift_log_rows
"""))
display(spark.sql(f"""
    SELECT appid, name, positive, ccu, last_load_date
    FROM silver_steam WHERE appid IN ({SAMPLE_APPIDS}) ORDER BY appid
"""))

In [0]:
dbutils.notebook.run("./02_raw_to_bronze", 3600,
                     {"load_type": "incremental", "date": TEST_DATE, "landing_path": TEST_FILE})
print("02_raw_to_bronze finished")

dbutils.notebook.run("./03_bronze_to_silver", 3600, {"date": TEST_DATE, "batch_id": ""})
print("03_bronze_to_silver finished")

In [0]:
print("genre column in bronze_steam:", "genre" in spark.table("bronze_steam").columns)

# 1. new column was logged
display(spark.sql("SELECT * FROM schema_drift_log"))

# 2. Bronze evolved and kept all 5 raw rows, including the bad value
display(spark.sql(f"""
    SELECT appid, name, positive, genre, load_date, source_file
    FROM bronze_steam WHERE load_date = DATE'{TEST_DATE}' ORDER BY appid
"""))

# 3. the row with the wrong type went to quarantine
display(spark.sql("SELECT appid, name, positive, reject_reason, load_date FROM silver_quarantine"))

# 4. Silver's real values were not touched
display(spark.sql(f"""
    SELECT appid, name, positive, ccu, last_load_date
    FROM silver_steam WHERE appid IN ({SAMPLE_APPIDS}) ORDER BY appid
"""))

# 5. both runs were logged
display(spark.sql("""
    SELECT layer, file_processed, status, rows_inserted, rows_updated,
           rows_skipped_unchanged, rows_quarantined
    FROM pipeline_execution_logs ORDER BY start_time DESC LIMIT 2
"""))

## Clean up test rows

In [0]:
spark.sql(f"""
    DELETE FROM bronze_steam
    WHERE load_date = DATE'{TEST_DATE}' AND source_file = '{TEST_FILE}'
""")
display(spark.sql("""
    SELECT (SELECT COUNT(*) FROM bronze_steam)      AS bronze_rows,
           (SELECT COUNT(*) FROM silver_steam)      AS silver_rows,
           (SELECT COUNT(*) FROM silver_quarantine) AS quarantine_rows,
           (SELECT COUNT(*) FROM schema_drift_log)  AS drift_log_rows
"""))